# vitok 01 — Data + tokenizers (CPU)

**Settings:** Accelerator = None (CPU), Internet = On. Attach the `vitok-code` dataset, or set `VITOK_REPO`.

Run with **Save Version → Save & Run All**. When it finishes, create a Kaggle Dataset named **`vitok-data`** from this notebook's output. Notebook 02 reads it.

Plan steps covered: 2 (data), 3 (tokenizers, Gate 1), minimal pairs (step 6 input).

In [1]:
VITOK_REPO = ""  # e.g. "https://github.com/<you>/vitok.git"; leave empty when the vitok-code dataset is attached
OUT = "/kaggle/working/vitok-data"
# PRETRAIN_BYTES = 1e10   # UTF-8 bytes of pretraining text (enough for d10 = 1B bpe-nfc tokens incl. packing loss)
# TOK_TRAIN_BYTES = 5e8
# VOCAB_SIZES = [16000, 32000]  # 32k only to check Gate 1 (plan step 3)
# --- TEST ---
PRETRAIN_BYTES = 2e8
TOK_TRAIN_BYTES = 5e7
VOCAB_SIZES = [16000]
# --- TEST ---
TRANSITION = 0.9
SUPERBPE_COMMIT = "bbd09768fc28a875cef48e6bdd66e3a17454628e"

In [2]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

CODE = Path("/tmp/vitok")
bundles = list(Path("/kaggle/input").rglob("pyproject.toml"))
bundles = [p.parent for p in bundles if (p.parent / "src" / "vitok").exists()]
if bundles:
    shutil.copytree(bundles[0], CODE, dirs_exist_ok=True)
else:
    assert VITOK_REPO, "attach the vitok-code dataset or set VITOK_REPO"
    sh(f"git clone --depth 1 {VITOK_REPO} {CODE}")
sh(f"pip install -q -e {CODE}")

$ pip install -q -e /tmp/vitok


In [3]:
# Build SuperBPE's forked `tokenizers` FIRST (needs Rust, ~10 min) so a build failure shows up before the long download.
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = f"{Path.home()}/.cargo/bin:" + os.environ["PATH"]
sh("git clone https://github.com/PythonNut/superbpe /tmp/superbpe")
sh(f"cd /tmp/superbpe && git checkout {SUPERBPE_COMMIT} && git submodule update --init --depth 1")
sh("pip install -q /tmp/superbpe/tokenizers_superbpe/bindings/python")

$ curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal


info: downloading installer
info: profile set to minimal
info: default host tuple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-09-03 for version 1.98.1 (48a229cea 2026-09-01)
info: downloading 3 components



  stable-x86_64-unknown-linux-gnu installed - rustc 1.98.1 (48a229cea 2026-09-01)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown-linux-gnu = None
cargo:rerun-if-env-changed=CC_x86_64_unknown_linux_gnu
CC_x86_64_unknown_linux_gnu = None
cargo:rerun-if-env-changed=HOST_CC
HOST_CC = None
cargo:rerun-if-env-changed=CC
CC = None
cargo:rerun-if-env-changed=CC_ENABLE_DEBUG_OUTPUT
cargo:rerun-if-env-changed=CRATE_CC_NO_DEFAULTS
CRATE_CC_NO_DEFAULTS = None
cargo:rerun-if-env-changed=CFLAGS
CFLAGS = None
cargo:rerun-if-env-changed=HOST_CFLAGS
HOST_CFLAGS = N

info: default toolchain set to stable-x86_64-unknown-linux-gnu
Cloning into '/tmp/superbpe'...


$ cd /tmp/superbpe && git checkout bbd09768fc28a875cef48e6bdd66e3a17454628e && git submodule update --init --depth 1


Updating files: 100% (669/669), done.
Note: switching to 'bbd09768fc28a875cef48e6bdd66e3a17454628e'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at bbd0976 fix bug caused by ByteLevel being imported from two different places
Submodule 'tokenizers-superbpe' (https://github.com/alisawuffles/tokenizers-superbpe.git) registered for path 'tokenizers_superbpe'
Cloning into '/tmp/superbpe/tokenizers_superbpe'...


Submodule path 'tokenizers_superbpe': checked out '757f2a55c0820ed47064e1fe473deea39b7b611b'
$ pip install -q /tmp/superbpe/tokenizers_superbpe/bindings/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.1 which is incompatible.


In [4]:
# Step 2: FineWeb-2 vie_Latn -> tok_train.txt, pretraining shards, val shard, test.jsonl, syllables.json
sh(f"python -m vitok.data --out {OUT} --pretrain-bytes {PRETRAIN_BYTES} --tok-train-bytes {TOK_TRAIN_BYTES} --cache-dir /tmp/hf_raw")
shutil.move(f"{OUT}/tok_train.txt", "/tmp/tok_train.txt")  # keep the output dataset small
sh(f"python -m vitok.minimal_pairs --test {OUT}/test.jsonl --syllables {OUT}/syllables.json --out {OUT}/minimal_pairs.jsonl")

$ python -m vitok.data --out /kaggle/working/vitok-data --pretrain-bytes 200000000.0 --tok-train-bytes 50000000.0 --cache-dir /tmp/hf_raw
{
  "tok_train_bytes": 50003457,
  "tok_train_docs": 7489,
  "pretrain_bytes": 200004809,
  "pretrain_chars": 151682405,
  "pretrain_docs": 37930,
  "train_shards": 1,
  "test_docs": 2000
}
$ python -m vitok.minimal_pairs --test /kaggle/working/vitok-data/test.jsonl --syllables /kaggle/working/vitok-data/syllables.json --out /kaggle/working/vitok-data/minimal_pairs.jsonl
wrote 3000 pairs to /kaggle/working/vitok-data/minimal_pairs.jsonl
  nhiều -> nhiêu: Cách làm đẹp da mặt tự nhiên tại nhà với cà chua, nha đam, mật ong… được rất nhi
  thể -> thề: Nguồn thạch tín có thề từ nước, thức ăn và thuốc.
  Là -> Lạ: Lạ loại sơn lý tưởng để thi công sân tennis trên nền xi măng hay bê tông.
  với -> vợi: Nếu tổng vốn sở hữu của nhà đầu tư nước ngoài dưới 51%, Doanh Nghiệp thực hiện c
  máy -> may: Tại thời điểm chương trình bắt đầu có 2 sản phẩm niêm là chuột 

In [5]:
# Step 3: train BPE / SuperBPE x NFC / NFD, then measure compression (Gate 1)
for v in VOCAB_SIZES:
    k = f"{v // 1000}k"
    sh(f"python -m vitok.train_tokenizers --corpus /tmp/tok_train.txt --out {OUT}/tokenizers-{k} "
       f"--vocab-size {v} --transition {TRANSITION} --workdir /tmp/tok_work_{k}")
    sh(f"python -m vitok.compression --tokenizers {OUT}/tokenizers-{k} "
       f"--docs {OUT}/shards/shard_99999.parquet --out {OUT}/compression-{k}.json")

$ python -m vitok.train_tokenizers --corpus /tmp/tok_train.txt --out /kaggle/working/vitok-data/tokenizers-16k --vocab-size 16000 --transition 0.9 --workdir /tmp/tok_work_16k
Calling do_train_original()



Calling do_train_extend()
In do_train_extend()
Step 1: Add special tokens
Step 2: Compute alphabet
Length of word_to_id: 256
Printing 5 elements in word_to_id
Word: Ĳ, ID: 238
Word: ·, ID: 115
Word: ē, ID: 207
Word: ï, ID: 171
Word: A, ID: 32
Word: C, ID: 34
Word: O, ID: 46
Word: Ġ, ID: 220
Word: <, ID: 27
Word: Ü, ID: 152
Step 3: Tokenize words

Step 4: Count pairs in words

Length of queue: 9550
Step 5: Apply merges

Length of merges: 14144
Step 5: Do new merges
Skipping merge : ĠâĢľ because of : special-casing
Skipping merge : Ġ" because of : special-casing
Skipping merge : Ġ because of : special-casing
Skipping merge ĠnÃ³i: ĠâĢľ because of : special-casing
Skipping merge [sá»Ńa Ġ|Ġsá»ŃaĠmÃ£Ġnguá»ĵn]Ċ because it has 5 words

Calling do_train_original()



Calling do_train_extend()

In [6]:
# Step 3 checks: round-trip 1,000 val docs for every tokenizer; inspect superwords
import pyarrow.parquet as pq
from vitok.hf_tokenizer import HFTokenizer
from vitok.tokenizer_spec import CONDITIONS

docs = pq.read_table(f"{OUT}/shards/shard_99999.parquet").column("text").to_pylist()[:1000]
for v in VOCAB_SIZES:
    k = f"{v // 1000}k"
    for cond in CONDITIONS:
        tok = HFTokenizer.from_directory(f"{OUT}/tokenizers-{k}/{cond}")
        bad = sum(tok.decode(ids) != d for ids, d in zip(tok.encode(docs), docs))
        print(f"{k} {cond:10s} round-trip failures: {bad}/1000")
        assert bad == 0
    comp = json.load(open(f"{OUT}/compression-{k}.json"))
    print(k, "top superwords (super-nfc):", [w for w, _ in comp["super-nfc"]["top_superwords"]])
print("Viet tokens (bpe-nfd 16k):", HFTokenizer.from_directory(f"{OUT}/tokenizers-16k/bpe-nfd").tok.encode("Tiếng Việt").tokens)
print(json.load(open(f"{OUT}/stats.json")))
sh(f"du -sh {OUT}/*")

ModuleNotFoundError: No module named 'vitok'